In [ ]:
import os
import sys

if not os.path.exists("src") and not os.path.exists("../src"):
    !git clone https://github.com/RuoyuLi97/info7160-goemotions-emotion-classification.git
    os.chdir("info7160-goemotions-emotion-classification")

if os.path.exists("src"):
    sys.path.append(os.getcwd())
else:
    sys.path.append(os.path.abspath(".."))

import json
import pandas as pd
from collections import defaultdict

from src.metrics import compute_metrics

In [ ]:
def load_jsonl(path):
    data = []
    try:
        with open(path, "r") as f:
            for line in f:
                data.append(json.loads(line))
    except:
        print(f"Warning: {path} not found, using dummy data")
        data = [
            {"text": "I love this", "true_labels": ["joy"], "pred_labels": ["joy"]},
            {"text": "This is bad", "true_labels": ["anger"], "pred_labels": ["sadness"]},
        ]
    return pd.DataFrame(data)

In [ ]:
baseline_path = "outputs/baseline_preds.jsonl"
bert_path = "outputs/bert_preds.jsonl"

In [ ]:
df_base_raw = load_jsonl(baseline_path)
df_bert_raw = load_jsonl(bert_path)

label_list = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","neutral","optimism","pride",
    "realization","relief","remorse","sadness","surprise"
]

def multi_hot_to_labels(vec):
    return [label_list[i] for i, v in enumerate(vec) if v == 1]

df_base_raw["true_labels"] = df_base_raw["true_labels"].apply(multi_hot_to_labels)
df_base_raw["pred_labels"] = df_base_raw["pred_labels"].apply(multi_hot_to_labels)

df_bert_raw["true_labels"] = df_bert_raw["true_labels"].apply(multi_hot_to_labels)
df_bert_raw["pred_labels"] = df_bert_raw["pred_labels"].apply(multi_hot_to_labels)

assert isinstance(df_bert_raw["true_labels"].iloc[0], list)
assert isinstance(df_bert_raw["true_labels"].iloc[0][0], str)

df_base = df_base_raw.copy(deep=True)
df_bert = df_bert_raw.copy(deep=True)

df_base.head()

In [ ]:
import numpy as np

all_labels = set()
for df in [df_base_raw, df_bert_raw]:
    for labels in df["true_labels"]:
       all_labels.update([str(l) for l in labels])

label_list = sorted(list(all_labels))
label_to_idx = {l: i for i, l in enumerate(label_list)}

def to_multi_hot(label_lists):
    mat = np.zeros((len(label_lists), len(label_list)))
    for i, labels in enumerate(label_lists):
        for l in labels:
            l = str(l)
            if l in label_to_idx:
                mat[i][label_to_idx[l]] = 1
    return mat

y_true_base = to_multi_hot(df_base["true_labels"])
y_pred_base = to_multi_hot(df_base["pred_labels"])

y_true_bert = to_multi_hot(df_bert["true_labels"])
y_pred_bert = to_multi_hot(df_bert["pred_labels"])

In [ ]:
metrics_base = compute_metrics(y_true_base, y_pred_base)
metrics_bert = compute_metrics(y_true_bert, y_pred_bert)

print("Baseline:", metrics_base)
print("BERT:", metrics_bert)

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": list(metrics_base.keys()),
    "Baseline": list(metrics_base.values()),
    "BERT": list(metrics_bert.values())
})

comparison

In [ ]:
from sklearn.metrics import f1_score

per_label_f1_base = f1_score(y_true_base, y_pred_base, average=None)
per_label_f1_bert = f1_score(y_true_bert, y_pred_bert, average=None)

label_df = pd.DataFrame({
    "label": label_list,
    "baseline_f1": per_label_f1_base,
    "bert_f1": per_label_f1_bert
})

label_df.sort_values(by="bert_f1", ascending=False)

In [ ]:
top5_easy = label_df.sort_values(by="bert_f1", ascending=False).head(5)
top5_hard = label_df.sort_values(by="bert_f1").head(5)

top5_easy
top5_hard

In [ ]:
def is_wrong(true, pred):
    return set(true) != set(pred)

errors = df_bert_raw[
    df_bert_raw.apply(
        lambda row: is_wrong(row["true_labels"], row["pred_labels"]),
        axis=1
    )
].head(5)

errors[["text", "true_labels", "pred_labels"]]